Importing the necessary functions and loading the model 

In [1]:
from typing import Any, Dict

import hydra
import numpy as np
import omegaconf
import torch
import pytorch_lightning as pl
import torch.nn as nn
from torch.nn import functional as F
from torch_scatter import scatter
from tqdm import tqdm

from cdvae.common.utils import PROJECT_ROOT
from cdvae.common.data_utils import (
    EPSILON, cart_to_frac_coords, mard, lengths_angles_to_volume,
    frac_to_cart_coords, min_distance_sqr_pbc)
from cdvae.pl_modules.embeddings import MAX_ATOMIC_NUM
from cdvae.pl_modules.embeddings import KHOT_EMBEDDINGS

#added by Tsach
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.diffraction.xrd import XRDCalculator
#import Batch
from torch_geometric.data import Batch
xrd_calculator = XRDCalculator(wavelength='CuKa', symprec=0.1)
torch.set_printoptions(threshold=50000) # use this if you want to print the entire tensor


In [2]:
import time
import argparse
import torch

from tqdm import tqdm
from torch.optim import Adam
from pathlib import Path
from types import SimpleNamespace
from torch_geometric.data import Batch
from torch_geometric.data.separate import separate

#import a library that allows you to reload a module
from importlib import reload

from eval_utils import load_model

all_frac_coords_stack = []
all_atom_types_stack = []
frac_coords = []
num_atoms = []
atom_types = []
lengths = []
angles = []
input_data_list = []

#my code 
list_of_idxs = []
list_of_batchs = []

In [50]:
import importlib 
import eval_utils
importlib.reload(eval_utils)
from eval_utils import load_model
model_path = Path("/home/gridsan/tmackey/hydra/singlerun/2023-10-28/no_encode_intensity_concat_comp_concat_neg_mask_v3")
model, test_loader, cfg = load_model(model_path, True)

loader = test_loader

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/hydra/experimental/compose.py:16: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  warnings.warn(


  0%|          | 0/3785 [00:00<?, ?it/s]

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))


CrystDataModule(self.datasets={'train': {'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy train', 'path': '/home/gridsan/tmackey/cdvae/data/perov_5/train.csv', 'prop': 'heat_ref', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}, 'val': [{'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy val', 'path': '/home/gridsan/tmackey/cdvae/data/perov_5/val.csv', 'prop': 'heat_ref', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}], 'test': [{'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy test', 'path': '/home/gridsan/tmackey/cdvae/data/perov_5/test.csv', 'prop': 'heat_ref', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}]}, self.num_workers={'train': 0, 'val': 0, 'test': 0

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/torch_geometric/deprecation.py:13: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [4]:
model.to('cuda:0')

CDVAE(
  (encoder): DimeNetPlusPlusWrap(
    (rbf): BesselBasisLayer(
      (envelope): Envelope()
    )
    (sbf): SphericalBasisLayer(
      (envelope): Envelope()
    )
    (emb): EmbeddingBlock(
      (emb): Embedding(95, 128)
      (lin_rbf): Linear(in_features=6, out_features=128, bias=True)
      (lin): Linear(in_features=384, out_features=128, bias=True)
    )
    (output_blocks): ModuleList(
      (0): OutputPPBlock(
        (lin_rbf): Linear(in_features=6, out_features=128, bias=False)
        (lin_up): Linear(in_features=128, out_features=256, bias=True)
        (lins): ModuleList(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): Linear(in_features=256, out_features=256, bias=True)
          (2): Linear(in_features=256, out_features=256, bias=True)
        )
        (lin): Linear(in_features=256, out_features=256, bias=False)
      )
      (1): OutputPPBlock(
        (lin_rbf): Linear(in_features=6, out_features=128, bias=False)
        (lin

In [5]:
for idx, batch in enumerate(loader):
    print(idx)
    list_of_idxs.append(idx)
    list_of_batchs.append(batch)

idx = list_of_idxs[0]
batch = list_of_batchs[0]

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35


In [6]:
def new_dataloader_batch_processor(batch): 
    batch_reserve = batch
    xrd_int = batch_reserve[1]
    xrd_loc = batch_reserve[2]
    atom_spec = batch_reserve[3]
    batch = batch[0]
    return batch_reserve, xrd_int, xrd_loc, atom_spec, batch

In [7]:
batch_reserve, xrd_int, xrd_loc, atom_spec, batch = new_dataloader_batch_processor(batch)

In [8]:
batch_all_frac_coords = []
batch_all_atom_types = []
batch_frac_coords, batch_num_atoms, batch_atom_types = [], [], []
batch_lengths, batch_angles = [], []

In [9]:
_, _, z = model.encode(batch, xrd_int, xrd_loc, atom_spec)

In [10]:
#get the first true crystal structure 
print(batch)

Batch(edge_index=[2, 19564], y=[256, 1], frac_coords=[2651, 3], atom_types=[2651], lengths=[256, 3], angles=[256, 3], to_jimages=[19564, 3], num_atoms=[256], num_bonds=[256], num_nodes=2651, batch=[2651], ptr=[257])


First stage: use the multiple evals script to get outputs for a single crystal 

In [12]:
num_atoms = batch.num_atoms[[0]].to('cuda:0')
atom_spec = atom_spec[[0]].to('cuda:0')
z = z[[0]].to('cuda:0')

self = model

ld_kwargs = SimpleNamespace(n_step_each=10,
                                step_lr=1e-4,
                                min_sigma=0,
                                save_traj=False,
                                disable_bar=False)

num_evals = 10

for eval_idx in range(num_evals):
    # set force atom types to be true 
    force_num_atoms = True
    gt_num_atoms = num_atoms if force_num_atoms else None

    gt_atom_types = None

    # #print out the arguments into the langevin_dynamics function
    # print("z: ", z)
    # print("ld_kwargs: ", ld_kwargs)
    # print("gt_num_atoms: ", gt_num_atoms)
    # print("gt_atom_types: ", gt_atom_types)

    outputs = model.langevin_dynamics(
        z, ld_kwargs, gt_num_atoms, gt_atom_types, atom_spec)

    # collect sampled crystals in this batch.
    batch_frac_coords.append(outputs['frac_coords'].detach().cpu())
    batch_num_atoms.append(outputs['num_atoms'].detach().cpu())
    batch_atom_types.append(outputs['atom_types'].detach().cpu())
    batch_lengths.append(outputs['lengths'].detach().cpu())
    batch_angles.append(outputs['angles'].detach().cpu())
    if ld_kwargs.save_traj:
        batch_all_frac_coords.append(
            outputs['all_frac_coords'][::down_sample_traj_step].detach().cpu())
        batch_all_atom_types.append(
            outputs['all_atom_types'][::down_sample_traj_step].detach().cpu())

In [26]:
num_atoms = []

In [28]:
batch_frac_coords

[tensor([[0.0906, 0.3487, 0.2085],
         [0.5874, 0.3576, 0.4592],
         [0.5978, 0.3510, 0.9540],
         [0.5875, 0.8494, 0.7060],
         [0.5892, 0.8520, 0.2062],
         [0.0857, 0.8486, 0.9545],
         [0.0873, 0.3475, 0.7079],
         [0.0886, 0.8559, 0.4595]]),
 tensor([[0.0060, 0.5366, 0.0989],
         [0.2619, 0.0384, 0.3500],
         [0.7569, 0.0468, 0.8444],
         [0.0083, 0.0380, 0.5967],
         [0.5112, 0.5440, 0.5959],
         [0.5214, 0.0384, 0.0948],
         [0.7582, 0.5386, 0.3482],
         [0.2572, 0.5424, 0.8450]]),
 tensor([[0.2353, 0.4962, 0.8489],
         [0.9829, 0.9981, 0.7224],
         [0.2384, 0.5010, 0.3459],
         [0.4807, 0.0077, 0.4735],
         [0.9851, 0.0015, 0.2240],
         [0.7351, 0.5030, 0.5965],
         [0.7289, 0.4970, 0.0987],
         [0.4863, 0.9980, 0.9716]]),
 tensor([[0.0360, 0.8543, 0.1766],
         [0.7053, 0.3655, 0.8541],
         [0.2648, 0.8688, 0.7227],
         [0.9040, 0.8582, 0.4433],
         [0.11

In [30]:
frac_coords = []

In [31]:
frac_coords.append(torch.stack(batch_frac_coords, dim=0))

In [32]:
frac_coords

[tensor([[[0.0906, 0.3487, 0.2085],
          [0.5874, 0.3576, 0.4592],
          [0.5978, 0.3510, 0.9540],
          [0.5875, 0.8494, 0.7060],
          [0.5892, 0.8520, 0.2062],
          [0.0857, 0.8486, 0.9545],
          [0.0873, 0.3475, 0.7079],
          [0.0886, 0.8559, 0.4595]],
 
         [[0.0060, 0.5366, 0.0989],
          [0.2619, 0.0384, 0.3500],
          [0.7569, 0.0468, 0.8444],
          [0.0083, 0.0380, 0.5967],
          [0.5112, 0.5440, 0.5959],
          [0.5214, 0.0384, 0.0948],
          [0.7582, 0.5386, 0.3482],
          [0.2572, 0.5424, 0.8450]],
 
         [[0.2353, 0.4962, 0.8489],
          [0.9829, 0.9981, 0.7224],
          [0.2384, 0.5010, 0.3459],
          [0.4807, 0.0077, 0.4735],
          [0.9851, 0.0015, 0.2240],
          [0.7351, 0.5030, 0.5965],
          [0.7289, 0.4970, 0.0987],
          [0.4863, 0.9980, 0.9716]],
 
         [[0.0360, 0.8543, 0.1766],
          [0.7053, 0.3655, 0.8541],
          [0.2648, 0.8688, 0.7227],
          [0.9040, 

In [33]:
frac_coords = torch.cat(frac_coords, dim=1)

In [34]:
frac_coords

tensor([[[0.0906, 0.3487, 0.2085],
         [0.5874, 0.3576, 0.4592],
         [0.5978, 0.3510, 0.9540],
         [0.5875, 0.8494, 0.7060],
         [0.5892, 0.8520, 0.2062],
         [0.0857, 0.8486, 0.9545],
         [0.0873, 0.3475, 0.7079],
         [0.0886, 0.8559, 0.4595]],

        [[0.0060, 0.5366, 0.0989],
         [0.2619, 0.0384, 0.3500],
         [0.7569, 0.0468, 0.8444],
         [0.0083, 0.0380, 0.5967],
         [0.5112, 0.5440, 0.5959],
         [0.5214, 0.0384, 0.0948],
         [0.7582, 0.5386, 0.3482],
         [0.2572, 0.5424, 0.8450]],

        [[0.2353, 0.4962, 0.8489],
         [0.9829, 0.9981, 0.7224],
         [0.2384, 0.5010, 0.3459],
         [0.4807, 0.0077, 0.4735],
         [0.9851, 0.0015, 0.2240],
         [0.7351, 0.5030, 0.5965],
         [0.7289, 0.4970, 0.0987],
         [0.4863, 0.9980, 0.9716]],

        [[0.0360, 0.8543, 0.1766],
         [0.7053, 0.3655, 0.8541],
         [0.2648, 0.8688, 0.7227],
         [0.9040, 0.8582, 0.4433],
         [0.11

In [27]:
# collect sampled crystals for this z.
frac_coords.append(torch.stack(batch_frac_coords, dim=0))
num_atoms.append(torch.stack(batch_num_atoms, dim=0))
atom_types.append(torch.stack(batch_atom_types, dim=0))
lengths.append(torch.stack(batch_lengths, dim=0))
angles.append(torch.stack(batch_angles, dim=0))
if ld_kwargs.save_traj:
    all_frac_coords_stack.append(torch.stack(batch_all_frac_coords, dim=0))
    all_atom_types_stack.append(torch.stack(batch_all_atom_types, dim=0))

# print(batch)

# Save the ground truth structure
input_data_list = input_data_list + batch.to_data_list()

frac_coords = torch.cat(frac_coords, dim=1)
num_atoms = torch.cat(num_atoms, dim=1)
atom_types = torch.cat(atom_types, dim=1)
lengths = torch.cat(lengths, dim=1)
angles = torch.cat(angles, dim=1)
if ld_kwargs.save_traj:
    all_frac_coords_stack = torch.cat(all_frac_coords_stack, dim=2)
    all_atom_types_stack = torch.cat(all_atom_types_stack, dim=2)
    
input_data_batch = Batch.from_data_list(input_data_list)

In [40]:
recon_out_name = 'eval_recon.pt'

torch.save({
    'input_data_batch': input_data_batch,
    'frac_coords': frac_coords,
    'num_atoms': num_atoms,
    'atom_types': atom_types,
    'lengths': lengths,
    'angles': angles,
    'all_frac_coords_stack': all_frac_coords_stack,
    'all_atom_types_stack': all_atom_types_stack,
}, "/home/gridsan/tmackey/cdvae/scripts/eval_recon.pt")

TypeError: 'generator' object is not callable

In [41]:
def get_crystals_list(
        frac_coords, atom_types, lengths, angles, num_atoms):
    """
    args:
        frac_coords: (num_atoms, 3)
        atom_types: (num_atoms)
        lengths: (num_crystals)
        angles: (num_crystals)
        num_atoms: (num_crystals)
    """
    assert frac_coords.size(0) == atom_types.size(0) == num_atoms.sum()
    assert lengths.size(0) == angles.size(0) == num_atoms.size(0)

    start_idx = 0
    crystal_array_list = []
    for batch_idx, num_atom in enumerate(num_atoms.tolist()):
        cur_frac_coords = frac_coords.narrow(0, start_idx, num_atom)
        cur_atom_types = atom_types.narrow(0, start_idx, num_atom)
        cur_lengths = lengths[batch_idx]
        cur_angles = angles[batch_idx]

        crystal_array_list.append({
            'frac_coords': cur_frac_coords.detach().cpu().numpy(),
            'atom_types': cur_atom_types.detach().cpu().numpy(),
            'lengths': cur_lengths.detach().cpu().numpy(),
            'angles': cur_angles.detach().cpu().numpy(),
        })
        start_idx = start_idx + num_atom
    return crystal_array_list

In [42]:
def get_crystal_array_list(file_path, batch_idx=0):
    data = load_data(file_path)
    crys_array_list = get_crystals_list(
        data['frac_coords'][batch_idx],
        data['atom_types'][batch_idx],
        data['lengths'][batch_idx],
        data['angles'][batch_idx],
        data['num_atoms'][batch_idx])

    if 'input_data_batch' in data:
        batch = data['input_data_batch']
        if isinstance(batch, dict):
            true_crystal_array_list = get_crystals_list(
                batch['frac_coords'], batch['atom_types'], batch['lengths'],
                batch['angles'], batch['num_atoms'])
        else:
            true_crystal_array_list = get_crystals_list(
                batch.frac_coords, batch.atom_types, batch.lengths,
                batch.angles, batch.num_atoms)
    else:
        true_crystal_array_list = None

    return crys_array_list, true_crystal_array_list

In [43]:
from p_tqdm import p_map

In [44]:
from collections import Counter
import argparse
import os
import json

import numpy as np
from pathlib import Path
from tqdm import tqdm
from p_tqdm import p_map
from scipy.stats import wasserstein_distance

from pymatgen.core.structure import Structure
from pymatgen.core.composition import Composition
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.structure_matcher import StructureMatcher
from matminer.featurizers.site.fingerprint import CrystalNNFingerprint
from matminer.featurizers.composition.composite import ElementProperty
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.diffraction.xrd import XRDCalculator
xrd_calculator = XRDCalculator(wavelength='CuKa', symprec=0.1)

In [45]:
class RecEval(object):

    def __init__(self, pred_crys, gt_crys, stol=0.5, angle_tol=10, ltol=0.3): #modified by Tsach from the original values of stol=0.5, angle_tol=10, ltol=0.3
        assert len(pred_crys) == len(gt_crys)
        self.matcher = StructureMatcher(
            stol=stol, angle_tol=angle_tol, ltol=ltol)
        self.preds = pred_crys
        self.gts = gt_crys

    def get_match_rate_and_rms(self):
        def process_one(pred, gt, is_valid):
            if not is_valid:
                return None
            try:
                rms_dist = self.matcher.get_rms_dist(
                    pred.structure, gt.structure)
                rms_dist = None if rms_dist is None else rms_dist[0]
                return rms_dist
            except Exception:
                return None
        #define a function that gets the diffraction patterns for pred and gt and returns the RMSD between them
        def process_diff_pattern(pred, gt, is_valid):
            if not is_valid:
                return None
            try:
                #get the structures
                pred_structure = pred.structure
                gt_structure = gt.structure
                pred_pattern = xrd_calculator.get_pattern(pred_structure)
                gt_pattern = xrd_calculator.get_pattern(gt_structure)

                pred_adjusted_vector = np.zeros(256)
                minimum = min(256, len(pred_pattern.x))
                pred_adjusted_vector[:minimum] = pred_pattern.x[:minimum]

                gt_adjusted_vector = np.zeros(256)
                minimum = min(256, len(gt_pattern.x))
                gt_adjusted_vector[:minimum] = gt_pattern.x[:minimum]
                
                #calculate the RMSD between the two patterns
                rms_dist = np.sqrt(np.mean((pred_adjusted_vector - gt_adjusted_vector)**2))

                return rms_dist
            except Exception:
                return None    

        validity = [c.valid for c in self.preds]

        rms_dists = []
        evaluate_diff_pattern = args.compare_diffraction_patterns
        if evaluate_diff_pattern:
            diff_dists = []
        for i in tqdm(range(len(self.preds))):
            rms_dists.append(process_one(
                self.preds[i], self.gts[i], validity[i]))
            if evaluate_diff_pattern:
                diff_dists.append(process_diff_pattern(self.preds[i], self.gts[i], validity[i]))
        rms_dists = np.array(rms_dists)
        if evaluate_diff_pattern:
            diff_dists = np.array(diff_dists)
            average_diff_dist = diff_dists[diff_dists != None].mean()
            #print out all the diff dists
            print("diff_dists: ", diff_dists)
        else:
            average_diff_dist = None
        match_rate = sum(rms_dists != None) / len(self.preds)
        mean_rms_dist = rms_dists[rms_dists != None].mean()

        return {'match_rate': match_rate,
                'rms_dist': mean_rms_dist,
                'diff_dist': average_diff_dist}

    def get_metrics(self):
        return self.get_match_rate_and_rms()

In [47]:
from eval_utils import (
    smact_validity, structure_validity, CompScaler, get_fp_pdist,
    load_config, load_data, get_crystals_list, prop_model_eval, compute_cov)


In [49]:
crys_array_list, true_crystal_array_list = get_crystal_array_list(
    '/home/gridsan/tmackey/cdvae/scripts/eval_recon.pt')
pred_crys = p_map(lambda x: Crystal(x), crys_array_list)
gt_crys = p_map(lambda x: Crystal(x), true_crystal_array_list)

rec_evaluator = RecEval(pred_crys, gt_crys)

RuntimeError: [enforce fail at inline_container.cc:222] . file not found: archive/data.pkl

In [ ]:
recon_metrics = rec_evaluator.get_metrics()

In [164]:
#### inputs 
gt_num_atoms = num_atoms
gt_elements_input = atom_types
num_atoms, _, lengths, angles, composition_per_atom = model.decode_stats(
            z, gt_num_atoms, gt_elements=gt_elements_input)
composition_per_atom = composition_per_atom.to('cuda:0') # this can be removed in native code 
num_atoms = batch.num_atoms[[0]].to('cuda:0')  #this can be removed in native code 

# Create a range tensor and repeat it as before
range_tensor = torch.arange(len(num_atoms), device='cuda:0')
crystal_ids = torch.repeat_interleave(range_tensor, num_atoms)

# Convert atom_types into a mask
atom_mask = atom_types != 0

# For each unique crystal_id, get its corresponding indices in composition_per_atom
unique_crystal_ids, counts = torch.unique(crystal_ids, return_counts=True)

composition_per_atom = composition_per_atom + 1

start_idx = 0
for u_id, count in zip(unique_crystal_ids, counts):
    relevant_elements = atom_types[u_id][atom_mask[u_id]]
    mask = torch.ones_like(composition_per_atom[start_idx])
    mask *= (-10**6)
    mask[relevant_elements-1] = 1

    # Create additive mask
    additive_mask_for_normalization = mask 
    additive_mask_for_normalization[relevant_elements-1] *= 0.0001

    # Apply masks to the relevant segment of composition_per_atom
    composition_per_atom[start_idx:start_idx+count] *= mask
    composition_per_atom[start_idx:start_idx+count] += additive_mask_for_normalization

    # Update start index for next iteration
    start_idx += count

In [165]:
self = model

In [166]:
batch.num_atoms = batch.num_atoms.to('cuda:0')

In [167]:
noise_level = torch.randint(0, self.sigmas.size(0),
                                    (batch.num_atoms.size(0),),
                                    device=self.device)
used_sigmas_per_atom = self.sigmas[noise_level].repeat_interleave(
    batch.num_atoms, dim=0)

type_noise_level = torch.randint(0, self.type_sigmas.size(0),
                                (batch.num_atoms.size(0),),
                                device=self.device)
used_type_sigmas_per_atom = (
    self.type_sigmas[type_noise_level].repeat_interleave(
        batch.num_atoms, dim=0))

In [168]:
pred_composition_per_atom = composition_per_atom
pred_composition_probs = F.softmax(
            pred_composition_per_atom.detach(), dim=-1)

In [169]:
print(pred_composition_probs)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0

In [170]:
batch.atom_types = batch.atom_types.to('cuda:0')

In [171]:
atom_type_probs = (
    F.one_hot(batch.atom_types[[range(8)]] - 1, num_classes=MAX_ATOMIC_NUM) +
    pred_composition_probs)

In [172]:
rand_atom_types = torch.multinomial(
    atom_type_probs, num_samples=1).squeeze(1) + 1

In [173]:
rand_atom_types

tensor([31, 31, 31, 31, 52, 52, 31, 31], device='cuda:0')

In [174]:
batch.frac_coords = batch.frac_coords.to('cuda:0')
batch.num_atoms = batch.num_atoms.to('cuda:0')
batch.lengths = batch.lengths.to('cuda:0')
batch.angles = batch.angles.to('cuda:0')

In [175]:
(pred_num_atoms, pred_lengths_and_angles, pred_lengths, pred_angles,
        pred_composition_per_atom) = self.decode_stats(
            z, batch.num_atoms[[0]], batch.lengths[[0]], batch.angles[[0]], True, gt_elements)

In [176]:
cart_noises_per_atom = (
    torch.randn_like(batch.frac_coords[[8]]))
cart_coords = frac_to_cart_coords(
    batch.frac_coords[[8]], pred_lengths[[0]], pred_angles[[0]], batch.num_atoms[[0]])
cart_coords = cart_coords + cart_noises_per_atom
noisy_frac_coords = cart_to_frac_coords(
    cart_coords[[0]], pred_lengths[[0]], pred_angles[[0]], batch.num_atoms[[0]])

In [177]:
pred_cart_coord_diff, pred_atom_types = self.decoder(
    z, noisy_frac_coords[[0]], rand_atom_types[[range(8)]], batch.num_atoms[[0]], pred_lengths[[0]], pred_angles[[0]], gt_elements[[0]])

In [178]:
print(pred_atom_types)

tensor([[  0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,  -7.0912,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,  -7.7928,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0

The resultant vector should have extremely large negative numbers in all places where the value doesn't correspond to the correct elements 

Second stage: creating a function and verifying the function 

In [179]:
def composition_constraint(self, atom_types, num_atoms, composition_per_atom):
    """
    Restrict the probability distribution from which the atom types are randomly drawn 
    to only include the elements that are present in the crystal.

    atom_types: the atom types in the crystal
    num_atoms: the number of atoms in the crystal
    composition_per_atom: the probability distribution from which the atom types are randomly drawn 

    """

    # Create a range tensor and repeat it as before
    range_tensor = torch.arange(len(num_atoms), device='cuda:0')
    crystal_ids = torch.repeat_interleave(range_tensor, num_atoms)

    # Convert atom_types into a mask
    atom_mask = atom_types != 0

    # For each unique crystal_id, get its corresponding indices in composition_per_atom
    unique_crystal_ids, counts = torch.unique(crystal_ids, return_counts=True)

    composition_per_atom = composition_per_atom + 1

    start_idx = 0
    for u_id, count in zip(unique_crystal_ids, counts):
        relevant_elements = atom_types[u_id][atom_mask[u_id]]
        mask = torch.ones_like(composition_per_atom[start_idx])
        mask *= (-10**6)
        mask[relevant_elements-1] = 1

        # Create additive mask
        additive_mask_for_normalization = mask 
        additive_mask_for_normalization[relevant_elements-1] *= 0.0001

        # Apply masks to the relevant segment of composition_per_atom
        composition_per_atom[start_idx:start_idx+count] *= mask
        composition_per_atom[start_idx:start_idx+count] += additive_mask_for_normalization

        # Update start index for next iteration
        start_idx += count
    
    return composition_per_atom

In [180]:
gt_num_atoms = num_atoms
num_atoms, _, lengths, angles, composition_per_atom = model.decode_stats(
            z, gt_num_atoms, gt_elements=gt_elements_input)
composition_per_atom = composition_per_atom.to('cuda:0') # this can be removed in native code 
num_atoms = batch.num_atoms[[0]].to('cuda:0')  #this can be removed in native code 

composition_constraint(model, atom_types, num_atoms, composition_per_atom)

tensor([[-2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
          3.6399e-04, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06,  3.2929e-04, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.

Third stage: ensuring that the function works in all contexts in which it is applied. 

Because I validated the performance of the code in a workflow modeled after the forward pass, and the other situations in which the code would be used (langevin dynamics) are highly similar to the forward pass (and I would modularize it into the decode states and decode functions themselves), I don't think further notebook code is neccesary for this third stage in this particular case. 

However, I will take the following steps to verify performance "in-vivo": 
1. For the first few steps / epochs, I will have the code print out the results from the composition constraint ("before/after") that kind of thing. I will do this in the decoder stats function and before and after the decode call at the end of the forward pass. For the langevin dynamics, I will just print it out for every batch because there's less of a risk of slowing things down or taking up memory. After verifying, I'll save the outputs for posterity and get rid of the print statements. 